# Full Dataset -- Preprocessing

Tokenises and encodes the full WikiQA dataset -- **all Label=0 and Label=1 rows** -- with no correct-answer filter. The processed files are saved to `data/processed_full/` and consumed by `02_seq2seq_basic.ipynb`.

**Environments:** runs locally (loads from `data/raw/`) and in Google Colab (downloads from HuggingFace). In Colab, use a GPU runtime (Runtime > Change runtime type > T4 GPU).

In [3]:
import sys
import re
import json
from collections import Counter
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess

    subprocess.run(["pip", "install", "-q", "datasets", "nltk"], check=True)

import nltk
import pandas as pd

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

if IN_COLAB:
    PROCESSED_DIR = Path("processed_full")
else:
    PROCESSED_DIR = Path("../../data/processed_full")

PROCESSED_DIR.mkdir(exist_ok=True)

MAX_Q_LEN = 20
MAX_A_LEN = 30
MIN_FREQ = 2

PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3


def tokenize(text: str) -> list[str]:
    """
    Converts raw text to a list of lowercase word tokens.
    Strips all characters except letters, digits, apostrophes and question marks,
    then splits using NLTK word_tokenize.
    @param text: Raw input string.
    @return: List of lowercase string tokens.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9'? ]", " ", text)
    return nltk.word_tokenize(text)


def filterByLength(dataFrame: pd.DataFrame) -> pd.DataFrame:
    """
    Removes Q&A pairs whose question or answer exceeds the configured length limits.
    Applies MAX_Q_LEN to questions and MAX_A_LEN to answers.
    @param dataFrame: DataFrame with 'q_tokens' and 'a_tokens' columns.
    @return: Filtered DataFrame with index reset.
    """
    dataFrame = dataFrame[dataFrame["q_tokens"].map(len) <= MAX_Q_LEN]
    dataFrame = dataFrame[dataFrame["a_tokens"].map(len) <= MAX_A_LEN]
    return dataFrame.reset_index(drop=True)


if IN_COLAB:
    from datasets import load_dataset

    ds = load_dataset("wiki_qa")

    def loadFromHuggingFace(splitName: str) -> pd.DataFrame:
        """
        Loads a WikiQA split from HuggingFace datasets and applies tokenisation and length filtering.
        @param splitName: HuggingFace split name ('train', 'validation', or 'test').
        @return: Filtered DataFrame with 'question', 'answer', 'q_tokens', and 'a_tokens' columns.
        """
        rows = [
            {"question": r["question"], "answer": r["answer"]} for r in ds[splitName]
        ]
        dataFrame = pd.DataFrame(rows)
        dataFrame["q_tokens"] = dataFrame["question"].map(tokenize)
        dataFrame["a_tokens"] = dataFrame["answer"].map(tokenize)
        return filterByLength(dataFrame)

    trainDf = loadFromHuggingFace("train")
    devDf = loadFromHuggingFace("validation")
    testDf = loadFromHuggingFace("test")

else:
    RAW_DIR = Path("../../data/raw")

    def loadSplit(path: Path) -> pd.DataFrame:
        """
        Loads a WikiQA TSV file and applies tokenisation and length filtering.
        Includes all rows regardless of label (no Label=1 filter).
        @param path: Absolute path to the TSV file.
        @return: Filtered DataFrame with 'question', 'answer', 'q_tokens', and 'a_tokens' columns.
        """
        dataFrame = pd.read_csv(path, sep="\t")
        dataFrame = dataFrame[["Question", "Sentence"]].rename(
            columns={"Question": "question", "Sentence": "answer"}
        )
        dataFrame["q_tokens"] = dataFrame["question"].map(tokenize)
        dataFrame["a_tokens"] = dataFrame["answer"].map(tokenize)
        return filterByLength(dataFrame)

    trainDf = loadSplit(RAW_DIR / "WikiQA-train.tsv")
    devDf = loadSplit(RAW_DIR / "WikiQA-dev.tsv")
    testDf = loadSplit(RAW_DIR / "WikiQA-test.tsv")

print(
    f"Q&A pairs -- train: {len(trainDf):,}  dev: {len(devDf):,}  test: {len(testDf):,}"
)

Q&A pairs -- train: 16,206  dev: 2,194  test: 4,934


In [4]:
tokenCounts = Counter()
for dataFrame in (trainDf, devDf, testDf):
    for tokens in dataFrame["q_tokens"]:
        tokenCounts.update(tokens)
    for tokens in dataFrame["a_tokens"]:
        tokenCounts.update(tokens)

specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
vocab = specials + [w for w, c in tokenCounts.most_common() if c >= MIN_FREQ]
tokenToIdx = {t: i for i, t in enumerate(vocab)}

print(f"Vocabulary size: {len(tokenToIdx):,}")


def encode(tokens: list[str]) -> list[int]:
    """
    Converts a list of string tokens to an integer ID sequence with SOS/EOS.
    @param tokens: List of string tokens to encode.
    @return: List of integer token IDs with SOS at index 0 and EOS at the end.
    """
    return [SOS_IDX] + [tokenToIdx.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]


for dataFrame in (trainDf, devDf, testDf):
    dataFrame["q_ids"] = dataFrame["q_tokens"].map(encode)
    dataFrame["a_ids"] = dataFrame["a_tokens"].map(encode)

# Save vocabulary mapping and encoded splits as JSON (version-independent format)
with open(PROCESSED_DIR / "token2idx.json", "w") as f:
    json.dump(tokenToIdx, f)

for name, dataFrame in [("train", trainDf), ("dev", devDf), ("test", testDf)]:
    with open(PROCESSED_DIR / f"{name}.json", "w") as f:
        json.dump(
            {
                "q_ids": dataFrame["q_ids"].tolist(),
                "a_ids": dataFrame["a_ids"].tolist(),
            },
            f,
        )

print(f"Saved to {PROCESSED_DIR}")

Vocabulary size: 17,956
Saved to ../../data/processed_full
